<a href="https://colab.research.google.com/github/Swaglord936/Projects/blob/main/Credit_Score_(Reconstruccion).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import re
import datetime as dt
import unicodedata
import pandas as pd
import numpy as np

# ============================================
# RUTAS
# ============================================

INPUTS = {
    "cxc": r"C:\OneDrive\OneDrive - Jockey Plaza\Documentos\Score Riesgo Locatarios\Inputs Dataset Score vf\Feed final (csv)\Estados de Cuenta (CxC).csv",
    "fact": r"C:\OneDrive\OneDrive - Jockey Plaza\Documentos\Score Riesgo Locatarios\Inputs Dataset Score vf\Feed final (csv)\Facturacion 2023-2025.csv",
    "prom": r"C:\OneDrive\OneDrive - Jockey Plaza\Documentos\Score Riesgo Locatarios\Inputs Dataset Score vf\Feed final (csv)\Promedio de pago 2025.csv",
    "ventas": r"C:\OneDrive\OneDrive - Jockey Plaza\Documentos\Score Riesgo Locatarios\Inputs Dataset Score vf\Feed final (csv)\Ventas.csv",
    "sentinel": r"C:\OneDrive\OneDrive - Jockey Plaza\Documentos\Score Riesgo Locatarios\Inputs Dataset Score vf\Feed final (csv)\Sentinel.csv",
    # Optional:
    # "ra": r"C:\Onedrive\OneDrive - Jockey Plaza\Documentos\Score Riesgo Locatarios\Dataset 2\Renta variable.csv",
}

OUTPUT_DIR = r"C:\OneDrive\OneDrive - Jockey Plaza\Documentos\Score Riesgo Locatarios\Inputs Dataset Score vf\Feed final (csv)\Output vf"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================
# CONFIG – Grupos, Bandas, Pesos
# ============================================

CONFIG = {
    "rent_related_groups": [
        "AGUA",
        "ENERGIA",
        "ENERGÍA",
        "FONDO RENTA MÍNIMA",
        "FONDO RENTA MINIMA",
        "FONDO RENTA PORCENTUAL",
        "FONDO RENTA PORECENTUAL",
        "GASTOS COMUNES",
        "GASTOS COMUNES TOTTUS",
        "RENTA MINIMA",
        "RENTA MÍNIMA",
    ],
    "core_rent_groups": [
        "RENTA MINIMA",
        "RENTA MÍNIMA",
        "RENTA PORCENTUAL",
        "GASTOS COMUNES",
        "FONDO RENTA MINIMA",
        "FONDO RENTA MÍNIMA",
        "FONDO RENTA PORCENTUAL",
        "FONDO RENTA PORECENTUAL",
    ],
    "bands": {
        "occ_cost": [0.10, 0.15, 0.20, 0.25],
        "promedio_pago": [15, 30, 60, 90],
        "ar_ratio": [0.5, 1.0, 2.0, 3.0],
    },
    "weights": {
        "promedio_pago": 0.20,
        "ar_ratio": 0.25,
        "garantia_meses": 0.20,
        "occ_cost": 0.25,
        "sentinel": 0.10,
    },
    "final_band_cutoffs": [35, 55, 75, 90],
}

BAND_SCORE = {"AA": 100, "A": 80, "B": 60, "CC": 40, "C": 20}

SPANISH_MONTHS = {
    "enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6,
    "julio": 7, "agosto": 8, "septiembre": 9, "setiembre": 9, "octubre": 10,
    "noviembre": 11, "diciembre": 12,
    "ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6,
    "jul": 7, "ago": 8, "sep": 9, "set": 9, "oct": 10, "nov": 11, "dic": 12,
}

# ============================================
# UTILIDADES
# ============================================

def read_any_csv(path: str) -> pd.DataFrame:
    encs = ["utf-8-sig", "latin-1", "cp1252"]
    seps = [";", "\t", ",", "|"]
    last_err = None

    for e in encs:
        for s in seps:
            try:
                df = pd.read_csv(path, encoding=e, sep=s, dtype=str, low_memory=False)
                if df.shape[1] <= 1:
                    continue

                clean_cols = []
                for c in df.columns:
                    c2 = str(c).replace("'", "").replace('"', "").strip()
                    clean_cols.append(c2)
                df.columns = clean_cols

                # Drop junk unnamed columns
                df = df.loc[:, ~df.columns.str.contains(r"^Unnamed", case=False, na=False)].copy()

                print(f"Loaded: {os.path.basename(path)} | shape={df.shape}")
                print("CLEANED COLUMN NAMES:", df.columns.tolist())
                return df
            except Exception as ex:
                last_err = ex
                continue

    raise RuntimeError(f"Could not read {path}. Last error: {last_err}")


def normalize_upper(s):
    if pd.isna(s):
        return None

    s = str(s).strip().upper()
    s = s.replace("'", "").replace('"', "")
    s = re.sub(r"[\t\n\r]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))

    s = s.replace(".", " ").replace(",", " ").replace("-", " ").replace("/", " ")
    s = re.sub(r"\s+", " ", s).strip()

    replacements = {
        r"\bS A C\b": "SOCIEDAD ANONIMA CERRADA",
        r"\bSAC\b": "SOCIEDAD ANONIMA CERRADA",
        r"\bS A\b": "SOCIEDAD ANONIMA",
        r"\bSA\b": "SOCIEDAD ANONIMA",
        r"\bS A A\b": "SOCIEDAD ANONIMA ABIERTA",
        r"\bSAA\b": "SOCIEDAD ANONIMA ABIERTA",
        r"\bCIA\b": "COMPANIA",
        "&": "Y",
    }
    for pat, repl in replacements.items():
        s = re.sub(pat, repl, s)

    tokens = s.split()
    unique_tokens = []
    for t in tokens:
        if not unique_tokens or unique_tokens[-1] != t:
            unique_tokens.append(t)
    s = " ".join(unique_tokens)

    blacklist = {"S", "A", "C", "COMP", "SOC", "ANON", "DE", "DEL", "LA", "EL", "LOS", "LAS"}
    filtered = [t for t in s.split() if t not in blacklist]
    if filtered:
        s = " ".join(filtered)

    s = re.sub(r"\s+", " ", s).strip()
    return s


def canonical_name(s):
    if pd.isna(s):
        return None

    s = str(s).upper()
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[^A-Z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    legal = {
        "SA", "SAA", "SAC", "SRL", "EIRL", "SAAC", "SAC.", "LTD", "LTDA",
        "SOCIEDAD", "ANONIMA", "CERRADA", "ABIERTA", "COMPANIA", "CIA", "CIA.",
        "S", "A", "C", "R", "L",
    }

    tokens = [t for t in s.split() if t not in legal]
    if not tokens:
        return None
    return " ".join(sorted(tokens))


def normalize_group_key(s):
    if s is None or pd.isna(s):
        return ""
    s = str(s).strip().upper()
    nfkd = unicodedata.normalize("NFD", s)
    return "".join(ch for ch in nfkd if not unicodedata.combining(ch))


def parse_number_like(x):
    if x is None or pd.isna(x):
        return np.nan

    s = str(x).strip().upper()

    for t in [
        "S/.", "S/", "US$", "USD", "PEN", "SOLES", "DÓLARES", "DOLARES", "$"
    ]:
        s = s.replace(t, "")

    s = s.replace(" ", "")

    # Handle 1.234,56
    if "." in s and "," in s:
        s = s.replace(".", "").replace(",", ".")
    # Handle 1234,56
    elif "," in s:
        s = s.replace(",", ".")

    try:
        return float(s)
    except Exception:
        return np.nan


def to_month_start(x):
    if pd.isna(x):
        return pd.NaT

    if isinstance(x, (pd.Timestamp, dt.date, dt.datetime)):
        d = pd.Timestamp(x)
        return pd.Timestamp(d.year, d.month, 1)

    s = str(x).strip()

    for dayfirst in [True, False]:
        try:
            d = pd.to_datetime(s, dayfirst=dayfirst, errors="raise")
            return pd.Timestamp(d.year, d.month, 1)
        except Exception:
            pass

    m = re.match(r"^([A-Za-záéíóúñÑ]{2,10})[-/](\d{2,4})$", s)
    if m:
        mm = SPANISH_MONTHS.get(m.group(1).lower())
        yy = int(m.group(2))
        yy = yy + 2000 if yy < 100 else yy
        if mm:
            return pd.Timestamp(yy, mm, 1)

    m2 = re.match(r"^\s*(\d{4})[-/](\d{1,2})\s*$", s)
    if m2:
        return pd.Timestamp(int(m2.group(1)), int(m2.group(2)), 1)

    return pd.NaT


def to_month_end(x):
    d = to_month_start(x)
    if pd.isna(d):
        return pd.NaT
    return d + pd.offsets.MonthEnd(0)


def extract_fecha(df: pd.DataFrame):
    cols = {c.lower(): c for c in df.columns}
    for k in [
        "fecha",
        "fecha emision",
        "fecha emisión",
        "fecha factura",
        "fecha facturacion",
        "fecha emisión factura",
        "periodo",
        "mes",
        "mes facturado",
        "emision",
    ]:
        if k in cols:
            return df[cols[k]].map(to_month_start)
    return pd.Series(pd.NaT, index=df.index)


def band_ascending(x, cfg):
    if pd.isna(x):
        return None
    c1, c2, c3, c4 = cfg
    if x < c1:
        return "AA"
    if x < c2:
        return "A"
    if x < c3:
        return "B"
    if x < c4:
        return "CC"
    return "C"


def final_band(x, cutoffs):
    if pd.isna(x):
        return None
    c1, c2, c3, c4 = cutoffs
    if x >= c4:
        return "AA"
    if x >= c3:
        return "A"
    if x >= c2:
        return "B"
    if x >= c1:
        return "CC"
    return "C"


def convert_to_usd(amount, moneda, tc):
    val = parse_number_like(amount)
    if pd.isna(val):
        return np.nan

    m = str(moneda).upper() if not pd.isna(moneda) else ""
    tc_val = parse_number_like(tc)

    if "PEN" in m or "S/" in m or m.strip() == "S":
        if tc_val and tc_val > 0:
            return val / tc_val
        return np.nan

    return val


def find_group_col(cols: dict):
    for key in cols:
        low = key.lower()
        if "nom/grupo" in low or "nom grupo" in low or low == "grupo":
            return cols[key]
    return None


def build_nombre_comercial(df):
    if df.empty:
        return pd.DataFrame(columns=["Razon Social", "Nombre Comercial"])

    cols = {c.lower(): c for c in df.columns}

    rz_col = None
    for k in ["razón social", "razon social", "cliente"]:
        if k in cols:
            rz_col = cols[k]
            break
    if not rz_col:
        return pd.DataFrame(columns=["Razon Social", "Nombre Comercial"])

    nc_col = None
    for k in [
        "nombre comercial",
        "nombre fantasia",
        "nombre fantasía",
        "nombre comercial/fantasia",
        "nombre comercial / fantasia",
        "nombre fantasía",
        "nombre comercial / fantasía",
    ]:
        if k in cols:
            nc_col = cols[k]
            break
    if not nc_col:
        return pd.DataFrame(columns=["Razon Social", "Nombre Comercial"])

    out = df[[rz_col, nc_col]].copy()
    out["Razon Social"] = out[rz_col].map(normalize_upper)
    out["Nombre Comercial"] = out[nc_col].astype(str).str.strip().replace({"": np.nan})

    out = out[out["Nombre Comercial"].str.upper() != "ADMINISTRADORA JOCKEY PLAZA"]
    out = out.dropna(subset=["Nombre Comercial"])

    agg = (
        out.groupby("Razon Social")["Nombre Comercial"]
        .apply(lambda x: " | ".join(sorted(set(x))))
        .reset_index()
    )
    return agg


# ============================================
# BUILDERS
# ============================================

def build_facturacion(fact):
    if fact.empty:
        empty = pd.DataFrame()
        return (
            empty, empty, empty,
            {"by_name": pd.DataFrame(columns=["Razon Social", "garantia_monto_usd"]),
             "by_canon": pd.DataFrame(columns=["rz_canon", "garantia_monto_usd"])},
            empty, empty, empty
        )

    fact = fact.copy()
    cols = {c.lower(): c for c in fact.columns}

    rz = cols.get("razón social") or cols.get("razon social") or cols.get("cliente")
    mon = cols.get("moneda") or cols.get("divisa")
    tc = cols.get("tipo de cambio") or cols.get("tipo cambio") or cols.get("tc")
    grp = find_group_col(cols)

    fact["Razon Social"] = fact[rz].map(normalize_upper) if rz else "SIN-RAZON"
    fact["Fecha"] = extract_fecha(fact)

    amt = None
    for k in ["total usd", "importe total usd", "suma total usd", "suma total", "importe", "total", "monto"]:
        if k in cols:
            amt = cols[k]
            break
    if not amt:
        amt = fact.columns[-1]

    if amt.lower() == "total usd":
        fact["usd"] = fact[amt].map(parse_number_like)
    elif mon:
        fact["usd"] = [
            convert_to_usd(a, mo, fact.loc[i, tc] if tc in fact.columns else None)
            for i, (a, mo) in enumerate(zip(fact[amt], fact[mon]))
        ]
    else:
        fact["usd"] = fact[amt].map(parse_number_like)

    if grp:
        fact["NomGrupo"] = fact[grp].map(normalize_upper).fillna("")
    else:
        fact["NomGrupo"] = ""

    fact["NomGrupoKey"] = fact["NomGrupo"].map(normalize_group_key)
    fact["rz_canon"] = fact["Razon Social"].map(canonical_name)

    rr_keys = {normalize_group_key(x) for x in CONFIG["rent_related_groups"]}
    core_keys = {normalize_group_key(x) for x in CONFIG["core_rent_groups"]}
    rm_key = normalize_group_key("RENTA MINIMA")
    rp_key = normalize_group_key("RENTA PORCENTUAL")

    fact_m = (
        fact.groupby(["Razon Social", "Fecha"], as_index=False)["usd"]
        .sum()
        .rename(columns={"usd": "facturacion_usd"})
    )

    rent_related = fact[fact["NomGrupoKey"].isin(rr_keys)]
    rent_related_m = (
        rent_related.groupby(["Razon Social", "Fecha"], as_index=False)["usd"]
        .sum()
        .rename(columns={"usd": "rent_related_usd"})
    )

    core = fact[fact["NomGrupoKey"].isin(core_keys)]
    core_m = (
        core.groupby(["Razon Social", "Fecha"], as_index=False)["usd"]
        .sum()
        .rename(columns={"usd": "fact_rent_core_usd"})
    )

    rm = fact[fact["NomGrupoKey"] == rm_key]
    renta_min_m = (
        rm.groupby(["Razon Social", "Fecha"], as_index=False)["usd"]
        .sum()
        .rename(columns={"usd": "renta_minima_usd"})
    )

    rp_fact = fact[fact["NomGrupoKey"] == rp_key]
    rp_in_fact_m = (
        rp_fact.groupby(["Razon Social", "Fecha"], as_index=False)["usd"]
        .sum()
        .rename(columns={"usd": "rp_en_fact_usd"})
    )

    gar = fact[fact["NomGrupoKey"].str.contains("GARANT", na=False)].copy()
    gar["usd"] = gar["usd"].clip(lower=0)

    garantia_by_name = (
        gar.groupby("Razon Social", as_index=False)["usd"]
        .max()
        .rename(columns={"usd": "garantia_monto_usd"})
    )

    garantia_by_canon = (
        gar.dropna(subset=["rz_canon"])
        .groupby("rz_canon", as_index=False)["usd"]
        .max()
        .rename(columns={"usd": "garantia_monto_usd"})
    )

    garantia_tenant = {"by_name": garantia_by_name, "by_canon": garantia_by_canon}

    fact_grouped = (
        fact.groupby(["Razon Social", "Fecha", "NomGrupo", "NomGrupoKey"], as_index=False)["usd"]
        .sum()
    )

    return fact_m, rent_related_m, renta_min_m, garantia_tenant, core_m, rp_in_fact_m, fact_grouped


def build_ventas(ventas):
    if ventas.empty:
        return pd.DataFrame(columns=["Razon Social", "Fecha", "ventas_usd"])

    ventas = ventas.copy()
    cols = {c.lower(): c for c in ventas.columns}

    rz = None
    for k in ["razón social", "razon social", "cliente", "nombre comercial", "razon social"]:
        if k in cols:
            rz = cols[k]
            break

    ventas["Razon Social"] = ventas[rz].map(normalize_upper) if rz else "SIN-RAZON"
    ventas["Fecha"] = extract_fecha(ventas)

    mon = cols.get("moneda") or cols.get("divisa")
    tc = cols.get("tipo de cambio") or cols.get("tipo cambio") or cols.get("tc")

    amt = None
    for k in ["total usd", "importe", "total", "venta", "suma total", "monto", "importe sin igv"]:
        if k in cols:
            amt = cols[k]
            break
    if not amt:
        amt = ventas.columns[-1]

    if amt.lower() == "total usd":
        ventas["ventas_usd"] = ventas[amt].map(parse_number_like)
    elif mon:
        ventas["ventas_usd"] = [
            convert_to_usd(a, mo, ventas.loc[i, tc] if tc in ventas.columns else None)
            for i, (a, mo) in enumerate(zip(ventas[amt], ventas[mon]))
        ]
    else:
        ventas["ventas_usd"] = ventas[amt].map(parse_number_like)

    # Evitar doble conteo por ventas antiguas reportadas tarde:
    # si el mismo (Razon Social, Fecha) aparece más de una vez,
    # conservar solo la última ocurrencia cargada.
    ventas = ventas.reset_index(drop=False).rename(columns={"index": "__rowid"})
    ventas = ventas.sort_values(["Razon Social", "Fecha", "__rowid"])
    latest = ventas.groupby(["Razon Social", "Fecha"], as_index=False).last()

    return latest[["Razon Social", "Fecha", "ventas_usd"]]


def build_rvar(rvar):
    if rvar.empty:
        return pd.DataFrame(columns=["Razon Social", "Fecha", "renta_variable_usd"])

    rvar = rvar.copy()
    cols = {c.lower(): c for c in rvar.columns}

    rz = None
    for k in ["razón social", "razon social", "cliente"]:
        if k in cols:
            rz = cols[k]
            break

    rvar["Razon Social"] = rvar[rz].map(normalize_upper) if rz else "SIN-RAZON"
    rvar["Fecha"] = extract_fecha(rvar)

    mon = cols.get("moneda") or cols.get("divisa")
    tc = cols.get("tipo de cambio") or cols.get("tipo cambio") or cols.get("tc")

    amt = None
    for k in ["total usd", "suma total", "importe", "total", "monto"]:
        if k in cols:
            amt = cols[k]
            break
    if not amt:
        amt = rvar.columns[-1]

    if amt.lower() == "total usd":
        rvar["renta_variable_usd"] = rvar[amt].map(parse_number_like)
    elif mon:
        rvar["renta_variable_usd"] = [
            convert_to_usd(a, mo, rvar.loc[i, tc] if tc in rvar.columns else None)
            for i, (a, mo) in enumerate(zip(rvar[amt], rvar[mon]))
        ]
    else:
        rvar["renta_variable_usd"] = rvar[amt].map(parse_number_like)

    return rvar.groupby(["Razon Social", "Fecha"], as_index=False)["renta_variable_usd"].sum()


def build_cxc(cxc):
    if cxc.empty:
        return pd.DataFrame(columns=["Razon Social", "Fecha", "cxc_saldo_usd"])

    cxc = cxc.copy()

    tenant_col = None
    for c in cxc.columns:
        if "raz" in c.lower() and "social" in c.lower():
            tenant_col = c
            break
    if not tenant_col:
        tenant_col = cxc.columns[0]

    month_cols = [c for c in cxc.columns if c != tenant_col and pd.notna(to_month_start(c))]

    long = cxc.melt(
        id_vars=[tenant_col],
        value_vars=month_cols,
        var_name="FechaCol",
        value_name="Saldo",
    )
    long["Razon Social"] = long[tenant_col].map(normalize_upper)
    long["Fecha"] = long["FechaCol"].map(to_month_start)
    long["Saldo"] = long["Saldo"].map(parse_number_like)

    return (
        long.dropna(subset=["Fecha"])
        .groupby(["Razon Social", "Fecha"], as_index=False)["Saldo"]
        .sum()
        .rename(columns={"Saldo": "cxc_saldo_usd"})
    )


def build_prom(pagos, fact):
    if pagos.empty:
        return pd.DataFrame(columns=["Razon Social", "Fecha", "promedio_pago_dias"])

    pagos = pagos.copy()
    fact = fact.copy()

    cols_p = {c.lower(): c for c in pagos.columns}
    cols_f = {c.lower(): c for c in fact.columns}

    rz = None
    for k in ["razón social", "razon social", "cliente"]:
        if k in cols_p:
            rz = cols_p[k]
            break

    pagos["Razon Social"] = pagos[rz].map(normalize_upper) if rz else "SIN-RAZON"

    per_col = cols_p.get("periodo")
    if per_col:
        pagos["Fecha_Periodo"] = pagos[per_col].map(to_month_start)
    else:
        pagos["Fecha_Periodo"] = extract_fecha(pagos)

    fe_col = cols_p.get("fecha emision") or cols_p.get("fecha emisión") or cols_p.get("emision")
    if fe_col:
        pagos["Fecha_Emision"] = pd.to_datetime(pagos[fe_col], errors="coerce", dayfirst=True)
    else:
        pagos["Fecha_Emision"] = pagos["Fecha_Periodo"]

    corte_col = cols_p.get("corte")
    mes_pagado_col = cols_p.get("mes pagado")

    if corte_col:
        pagos["Fecha_Pago"] = pd.to_datetime(pagos[corte_col], errors="coerce", dayfirst=True)
    elif mes_pagado_col:
        pagos["Fecha_Pago"] = pagos[mes_pagado_col].map(to_month_end)
    else:
        pagos["Fecha_Pago"] = pagos["Fecha_Periodo"] + pd.offsets.MonthEnd(0)

    num_p = cols_p.get("numeracion")
    num_f = cols_f.get("numeracion")

    if num_p and num_f:
        grp = find_group_col(cols_f)
        if grp:
            fact["NomGrupo"] = fact[grp].map(normalize_upper).fillna("")
            fact["NomGrupoKey"] = fact["NomGrupo"].map(normalize_group_key)

            lookup = fact[[num_f, "NomGrupo", "NomGrupoKey"]].dropna(subset=[num_f]).copy()
            lookup["NumeracionKey"] = lookup[num_f].astype(str).str.strip()

            pagos["NumeracionKey"] = pagos[num_p].astype(str).str.strip()
            pagos = pagos.merge(
                lookup[["NumeracionKey", "NomGrupo", "NomGrupoKey"]],
                on="NumeracionKey",
                how="left"
            )
        else:
            pagos["NomGrupo"] = np.nan
            pagos["NomGrupoKey"] = np.nan
    else:
        pagos["NomGrupo"] = np.nan
        pagos["NomGrupoKey"] = np.nan

    pagos["lag_dias"] = (pagos["Fecha_Pago"] - pagos["Fecha_Emision"]).dt.days.abs()
    pagos = pagos[~pagos["NomGrupoKey"].astype(str).str.contains("GARANT", na=False)].copy()

    def group_weight(key):
        if not isinstance(key, str):
            return 3.0
        k = key.upper()
        if "AGUA" in k:
            return 5.0
        if "ENERGIA" in k or "ENERGÍA" in k:
            return 5.0
        if "FONDO" in k and "PORCENT" in k:
            return 6.0
        if "FONDO" in k and "MINIM" in k:
            return 6.0
        if "RENTA" in k and "MINIM" in k and "FONDO" not in k:
            return 8.0
        if "RENTA" in k and "PORCENT" in k and "FONDO" not in k:
            return 7.0
        if "GASTO" in k and "COMUN" in k:
            return 7.0
        return 3.0

    pagos["peso_grupo"] = pagos["NomGrupoKey"].map(group_weight)
    pagos["Fecha"] = pagos["Fecha_Periodo"]

    pagos_valid = pagos.dropna(subset=["Fecha", "lag_dias", "peso_grupo"]).copy()
    pagos_valid["lag_weighted"] = pagos_valid["lag_dias"] * pagos_valid["peso_grupo"]

    agg = (
        pagos_valid.groupby(["Razon Social", "Fecha"], as_index=False)
        .agg(
            lag_weighted_sum=("lag_weighted", "sum"),
            peso_sum=("peso_grupo", "sum"),
        )
    )
    agg["promedio_pago_dias"] = agg["lag_weighted_sum"] / agg["peso_sum"]

    return agg[["Razon Social", "Fecha", "promedio_pago_dias"]]


def build_sentinel(df):
    if df.empty:
        return pd.DataFrame(columns=["Razon Social", "Fecha", "sentinel_score"])

    df = df.copy()
    cols = {c.lower(): c for c in df.columns}

    rz_col = cols.get("razon social") or cols.get("razón social")
    fecha_col = cols.get("fecha")
    nor_col = cols.get("nor")

    col_act = cols.get("sem. act.") or cols.get("sem. act") or cols.get("sema act")
    col_pre = cols.get("sem. pre.") or cols.get("sem. pre") or cols.get("sema pre")
    col_12m = cols.get("sem. 12m") or cols.get("sem. 12 m") or cols.get("sema 12m")

    if not rz_col or not fecha_col:
        return pd.DataFrame(columns=["Razon Social", "Fecha", "sentinel_score"])

    df["Razon Social"] = df[rz_col].map(normalize_upper)
    df["Fecha"] = df[fecha_col].map(to_month_start)

    def clean_light(x):
        if pd.isna(x):
            return None
        x = str(x).strip().upper()
        x = unicodedata.normalize("NFD", x)
        return "".join(ch for ch in x if not unicodedata.combining(ch))

    df["light_act"] = df[col_act].map(clean_light) if col_act else None
    df["light_pre"] = df[col_pre].map(clean_light) if col_pre else None
    df["light_12m"] = df[col_12m].map(clean_light) if col_12m else None
    df["NOR"] = pd.to_numeric(df[nor_col], errors="coerce") if nor_col else np.nan

    temp = df[["NOR", "light_act"]].dropna(subset=["NOR", "light_act"]).copy()
    avg_by_light = temp.groupby("light_act")["NOR"].mean().to_dict()

    def est(light):
        light = (light or "").upper()
        if light in avg_by_light:
            return avg_by_light[light]
        if light in ("GRIS", "BLANCO"):
            return np.nan
        return np.nan

    rows = []
    for _, row in df.iterrows():
        rz = row["Razon Social"]
        f0 = row["Fecha"]

        score_now = row["NOR"] if not pd.isna(row["NOR"]) else est(row["light_act"])
        if not pd.isna(score_now):
            rows.append({"Razon Social": rz, "Fecha": f0, "sentinel_score": score_now})

        if row["light_pre"] not in (None, "GRIS", "BLANCO"):
            f_pre = f0 - pd.DateOffset(months=6)
            score_pre = est(row["light_pre"])
            if not pd.isna(score_pre):
                rows.append({"Razon Social": rz, "Fecha": f_pre, "sentinel_score": score_pre})

        if row["light_12m"] not in (None, "GRIS", "BLANCO"):
            f_12m = f0 - pd.DateOffset(months=12)
            score_12m = est(row["light_12m"])
            if not pd.isna(score_12m):
                rows.append({"Razon Social": rz, "Fecha": f_12m, "sentinel_score": score_12m})

    long = pd.DataFrame(rows)
    if long.empty:
        return pd.DataFrame(columns=["Razon Social", "Fecha", "sentinel_score"])

    out = []
    for tenant, g in long.groupby("Razon Social"):
        g = g.sort_values("Fecha").set_index("Fecha")
        g = g.resample("MS").ffill().bfill()
        g["Razon Social"] = tenant
        out.append(g.reset_index())

    final = pd.concat(out, ignore_index=True)
    return final[["Razon Social", "Fecha", "sentinel_score"]]


# ============================================
# PIPELINE
# ============================================

def run_pipeline():
    fact = read_any_csv(INPUTS["fact"])
    ventas = read_any_csv(INPUTS["ventas"])
    cxc_df = read_any_csv(INPUTS["cxc"])
    pagos = read_any_csv(INPUTS["prom"])
    sent = read_any_csv(INPUTS["sentinel"])
    rvar = read_any_csv(INPUTS["ra"]) if "ra" in INPUTS and os.path.exists(INPUTS["ra"]) else pd.DataFrame()

    nc_fact = build_nombre_comercial(fact)
    nc_ventas = build_nombre_comercial(ventas)

    nc_all = pd.concat([nc_fact, nc_ventas], ignore_index=True)
    if not nc_all.empty:
        nc_all = (
            nc_all.groupby("Razon Social")["Nombre Comercial"]
            .apply(lambda x: " | ".join(sorted(set(" | ".join(x).split(" | ")))))
            .reset_index()
        )
    else:
        nc_all = pd.DataFrame(columns=["Razon Social", "Nombre Comercial"])

    (
        fact_m,
        rent_related_m,
        renta_min_m,
        garantia_tenant,
        core_m,
        rp_in_fact_m,
        fact_grouped,
    ) = build_facturacion(fact)

    ventas_m = build_ventas(ventas)
    rvar_m = build_rvar(rvar)
    cxc_m = build_cxc(cxc_df)
    prom_m = build_prom(pagos, fact)
    sent_m = build_sentinel(sent)

    cutoff_min = pd.Timestamp(2023, 1, 1)

    def drop_before_2023(df):
        if "Fecha" not in df.columns:
            return df
        return df[df["Fecha"] >= cutoff_min].copy()

    fact_m = drop_before_2023(fact_m)
    rent_related_m = drop_before_2023(rent_related_m)
    renta_min_m = drop_before_2023(renta_min_m)
    core_m = drop_before_2023(core_m)
    rp_in_fact_m = drop_before_2023(rp_in_fact_m)
    ventas_m = drop_before_2023(ventas_m)
    rvar_m = drop_before_2023(rvar_m)
    cxc_m = drop_before_2023(cxc_m)
    prom_m = drop_before_2023(prom_m)
    sent_m = drop_before_2023(sent_m)

    # Remove YOY-only tenants
    fact_work = fact.copy()
    cols_fact = {c.lower(): c for c in fact_work.columns}
    sector_col = cols_fact.get("nom/sector")

    if sector_col:
        if "Razon Social" not in fact_work.columns:
            rz = cols_fact.get("razón social") or cols_fact.get("razon social") or cols_fact.get("cliente")
            fact_work["Razon Social"] = fact_work[rz].map(normalize_upper) if rz else "SIN-RAZON"

        fact_work["NomSector"] = (
            fact_work[sector_col].fillna("").astype(str).str.upper().str.strip()
        )

        invalid_sectors = {"YOY BOX PARK", ""}
        tenant_sector_sets = (
            fact_work.groupby("Razon Social")["NomSector"]
            .apply(lambda s: set(s.unique()))
        )

        tenants_yoy_only = [
            tenant for tenant, sectors in tenant_sector_sets.items()
            if sectors.issubset(invalid_sectors)
        ]

        if tenants_yoy_only:
            print("\nTenants REMOVED because they belong ONLY to YOY BOX PARK or blanks:")
            for t in tenants_yoy_only:
                print(" -", t)

            for dfname in [
                "fact_m", "rent_related_m", "renta_min_m", "core_m",
                "rp_in_fact_m", "fact_grouped", "ventas_m", "rvar_m",
                "cxc_m", "prom_m", "sent_m"
            ]:
                dfobj = locals()[dfname]
                if "Razon Social" in dfobj.columns:
                    locals()[dfname] = dfobj[~dfobj["Razon Social"].isin(tenants_yoy_only)].copy()

            garantia_by_name = garantia_tenant["by_name"]
            garantia_by_canon = garantia_tenant["by_canon"]

            garantia_by_name = garantia_by_name[
                ~garantia_by_name["Razon Social"].isin(tenants_yoy_only)
            ].copy()

            yoy_canon = fact_work[fact_work["Razon Social"].isin(tenants_yoy_only)].copy()
            yoy_canon["rz_canon"] = yoy_canon["Razon Social"].map(canonical_name)

            garantia_by_canon = garantia_by_canon[
                ~garantia_by_canon["rz_canon"].isin(yoy_canon["rz_canon"].dropna().unique())
            ].copy()

            garantia_tenant = {"by_name": garantia_by_name, "by_canon": garantia_by_canon}

    # Merge base
    base = ventas_m.merge(fact_m, on=["Razon Social", "Fecha"], how="outer")
    base = base.merge(rvar_m, on=["Razon Social", "Fecha"], how="left")
    base = base.merge(cxc_m, on=["Razon Social", "Fecha"], how="left")
    base = base.merge(prom_m, on=["Razon Social", "Fecha"], how="left")
    base = base.merge(rent_related_m, on=["Razon Social", "Fecha"], how="left")
    base = base.merge(renta_min_m, on=["Razon Social", "Fecha"], how="left")
    base = base.merge(core_m, on=["Razon Social", "Fecha"], how="left")
    base = base.merge(rp_in_fact_m, on=["Razon Social", "Fecha"], how="left")
    base = base.merge(sent_m, on=["Razon Social", "Fecha"], how="left")
    base = base.merge(nc_all, on="Razon Social", how="left")

    for c in [
        "ventas_usd", "facturacion_usd", "renta_variable_usd", "cxc_saldo_usd",
        "rent_related_usd", "renta_minima_usd", "fact_rent_core_usd",
        "rp_en_fact_usd", "garantia_monto_usd", "promedio_pago_dias",
        "sentinel_score"
    ]:
        if c in base.columns:
            base[c] = pd.to_numeric(base[c], errors="coerce")

    base["Fecha"] = pd.to_datetime(base["Fecha"], errors="coerce")
    base = base.dropna(subset=["Fecha"]).copy()
    base["days_in_month"] = base["Fecha"].dt.daysinmonth

    cutoff = pd.Timestamp(2025, 12, 31)
    base = base[base["Fecha"] <= cutoff].copy()

    # Guarantees by raw and canon
    garantia_by_name = garantia_tenant["by_name"]
    garantia_by_canon = garantia_tenant["by_canon"]

    base = base.merge(garantia_by_name, on="Razon Social", how="left")
    base["rz_canon"] = base["Razon Social"].map(canonical_name)
    base = base.merge(
        garantia_by_canon,
        on="rz_canon",
        how="left",
        suffixes=("", "_canon")
    )

    base["garantia_monto_usd"] = base["garantia_monto_usd"].fillna(base["garantia_monto_usd_canon"])
    if "garantia_monto_usd_canon" in base.columns:
        base = base.drop(columns=["garantia_monto_usd_canon"])

    base = base.sort_values(["Razon Social", "Fecha"]).copy()

    def ffill_bfill_if_any(s):
        if s.notna().any():
            return s.ffill().bfill()
        return s

    if "sentinel_score" in base.columns:
        base["sentinel_score"] = base.groupby("Razon Social")["sentinel_score"].transform(ffill_bfill_if_any)

    # Remove inactive tenants
    base["facturacion_usd"] = base["facturacion_usd"].fillna(0)

    inactive_tenants = []
    for tenant, df_t in base.groupby("Razon Social"):
        df_t = df_t.sort_values("Fecha")
        last3 = df_t.tail(3)
        if len(last3) == 3 and (last3["facturacion_usd"] == 0).all():
            inactive_tenants.append(tenant)

    if inactive_tenants:
        print("\nTenants excluded because LAST 3 months of facturación = 0:")
        for t in inactive_tenants:
            print(" -", t)
        base = base[~base["Razon Social"].isin(inactive_tenants)].copy()

    # Metrics
    base["rent_related_final_usd"] = (
        base["rent_related_usd"].fillna(0) + base["rp_en_fact_usd"].fillna(0)
    )

    base["ar_ratio"] = np.where(
        base["facturacion_usd"] > 0,
        base["cxc_saldo_usd"].fillna(0) / base["facturacion_usd"].fillna(0),
        np.nan
    )

    base["occ_cost"] = np.where(
        base["ventas_usd"] > 0,
        base["rent_related_final_usd"].fillna(0) / base["ventas_usd"].fillna(0),
        np.nan
    )

    base["garantia_meses"] = np.where(
        base["cxc_saldo_usd"] > 0,
        base["garantia_monto_usd"].fillna(0) / base["cxc_saldo_usd"],
        np.nan
    )
    base.loc[(base["cxc_saldo_usd"] == 0) & (base["garantia_monto_usd"] > 0), "garantia_meses"] = 999
    base.loc[(base["cxc_saldo_usd"] < 0) & (base["garantia_monto_usd"] > 0), "garantia_meses"] = 999
    base.loc[base["cxc_saldo_usd"] < 0, "garantia_meses"] = 999

    # Pre ingreso
    cols_check = [
        "ventas_usd", "occ_cost", "cxc_saldo_usd", "fact_rent_core_usd",
        "promedio_pago_dias", "ar_ratio"
    ]

    def mark_pre_ingreso(df):
        df = df.sort_values("Fecha").copy()
        df["is_empty"] = df[cols_check].apply(
            lambda r: all((pd.isna(v) or v == 0) for v in r),
            axis=1
        )
        try:
            first_real_idx = df.index[~df["is_empty"]][0]
        except IndexError:
            df["pre_ingreso"] = True
            return df.drop(columns=["is_empty"])

        df["pre_ingreso"] = False
        df.loc[df.index < first_real_idx, "pre_ingreso"] = df.loc[df.index < first_real_idx, "is_empty"]
        return df.drop(columns=["is_empty"])

    base = base.groupby("Razon Social", group_keys=False).apply(mark_pre_ingreso)

    # Bands
    b = CONFIG["bands"]
    base["band_occ_cost"] = base["occ_cost"].map(lambda x: band_ascending(x, b["occ_cost"]))
    base["band_promedio_pago"] = base["promedio_pago_dias"].map(lambda x: band_ascending(x, b["promedio_pago"]))
    base["band_ar_ratio"] = base["ar_ratio"].map(lambda x: band_ascending(x, b["ar_ratio"]))

    def band_sentinel(x):
        if pd.isna(x):
            return None
        if x >= 85:
            return "AA"
        if x >= 70:
            return "A"
        if x >= 50:
            return "B"
        if x >= 30:
            return "CC"
        return "C"

    def band_garantia(x):
        if pd.isna(x) or x <= 0:
            return "C"
        if x < 1:
            return "CC"
        if x < 2:
            return "B"
        if x < 3:
            return "A"
        return "AA"

    base["band_sentinel"] = base["sentinel_score"].map(band_sentinel)
    base["band_garantia_meses"] = base["garantia_meses"].map(band_garantia)

    # Score components
    w = CONFIG["weights"]

    base["score_pp"] = base["band_promedio_pago"].map(BAND_SCORE).fillna(0)
    base["score_ar"] = base["band_ar_ratio"].map(BAND_SCORE).fillna(0)
    base["score_gar"] = base["band_garantia_meses"].map(BAND_SCORE).fillna(0)
    base["score_occ"] = base["band_occ_cost"].map(BAND_SCORE).fillna(0)
    base["score_sent"] = base["band_sentinel"].map(BAND_SCORE).fillna(0)

    BANK_TENANTS = {
        "BANCO INTERNACIONAL PERU INTERBANK",
        "BANCO CREDITO PERU",
        "BANCO COMERCIO",
        "BANCO BBVA PERU",
        "SCOTIABANK PERU SOCIEDAD ANONIMA ABIERTA",
    }

    def redistribute(weights, missing_key):
        new_w = weights.copy()
        lost = new_w[missing_key]
        new_w[missing_key] = 0.0
        remaining_keys = [k for k in new_w if k != missing_key]
        total_remaining = sum(new_w[k] for k in remaining_keys)
        if total_remaining == 0:
            return new_w
        scale = (total_remaining + lost) / total_remaining
        for k in remaining_keys:
            new_w[k] *= scale
        return new_w

    def compute_row_score(row):
        if row["Razon Social"] in BANK_TENANTS:
            w_adj = {
                "pp": w["promedio_pago"] * (1 / 0.75),
                "ar": w["ar_ratio"] * (1 / 0.75),
                "gar": w["garantia_meses"] * (1 / 0.75),
                "sent": w["sentinel"] * (1 / 0.75),
                "occ": 0.0,
            }
            return (
                w_adj["pp"] * row["score_pp"] +
                w_adj["ar"] * row["score_ar"] +
                w_adj["gar"] * row["score_gar"] +
                w_adj["sent"] * row["score_sent"]
            )

        component_present = {
            "pp": not pd.isna(row["band_promedio_pago"]),
            "ar": not pd.isna(row["band_ar_ratio"]),
            "gar": not pd.isna(row["band_garantia_meses"]),
            "occ": not pd.isna(row["band_occ_cost"]),
            "sent": not pd.isna(row["band_sentinel"]),
        }
        missing_keys = [k for k, ok in component_present.items() if not ok]

        if len(missing_keys) == 0:
            return (
                w["promedio_pago"] * row["score_pp"] +
                w["ar_ratio"] * row["score_ar"] +
                w["garantia_meses"] * row["score_gar"] +
                w["occ_cost"] * row["score_occ"] +
                w["sentinel"] * row["score_sent"]
            )

        if len(missing_keys) == 1:
            mk = missing_keys[0]
            w_keys = {
                "pp": w["promedio_pago"],
                "ar": w["ar_ratio"],
                "gar": w["garantia_meses"],
                "occ": w["occ_cost"],
                "sent": w["sentinel"],
            }
            w_adj = redistribute(w_keys, mk)
            return (
                w_adj["pp"] * row["score_pp"] +
                w_adj["ar"] * row["score_ar"] +
                w_adj["gar"] * row["score_gar"] +
                w_adj["occ"] * row["score_occ"] +
                w_adj["sent"] * row["score_sent"]
            )

        return (
            w["promedio_pago"] * row["score_pp"] +
            w["ar_ratio"] * row["score_ar"] +
            w["garantia_meses"] * row["score_gar"] +
            w["occ_cost"] * row["score_occ"] +
            w["sentinel"] * row["score_sent"]
        )

    base["score_composite"] = base.apply(compute_row_score, axis=1)
    base["rating"] = base["score_composite"].map(
        lambda x: final_band(x, CONFIG["final_band_cutoffs"])
    )

    base.loc[base["pre_ingreso"], "score_composite"] = np.nan
    base.loc[base["pre_ingreso"], "rating"] = "pre ingreso"

    # PD
    PD_MAP = {
        "AA": 0.002,
        "A": 0.005,
        "B": 0.015,
        "CC": 0.040,
        "C": 0.120,
    }

    base["pd_annual"] = base["rating"].map(PD_MAP)
    base["pd_monthly"] = 1 - (1 - base["pd_annual"]) ** (1 / 12)

    # Outputs
    raw_cols = [
        "Fecha", "Razon Social", "Nombre Comercial",
        "ventas_usd", "renta_variable_usd",
        "facturacion_usd", "fact_rent_core_usd", "cxc_saldo_usd",
        "renta_minima_usd", "garantia_monto_usd",
        "rent_related_usd", "rp_en_fact_usd", "rent_related_final_usd",
        "promedio_pago_dias", "ar_ratio", "occ_cost", "garantia_meses",
        "sentinel_score", "band_sentinel",
        "band_occ_cost", "band_promedio_pago", "band_ar_ratio", "band_garantia_meses",
        "score_composite", "rating", "pd_annual", "pd_monthly",
    ]
    raw_cols = [c for c in raw_cols if c in base.columns]

    raw_path = os.path.join(OUTPUT_DIR, "tenant_month_ratings.csv")
    base[raw_cols].sort_values(["Razon Social", "Fecha"]).to_csv(
        raw_path, index=False, encoding="utf-8-sig"
    )

    nice = base.copy()
    nice["Costo de Ocupación (%)"] = (nice["occ_cost"] * 100).round(1)

    nice_es = nice.rename(columns={
        "Razon Social": "Razón Social",
        "ventas_usd": "Ventas (USD)",
        "rent_related_final_usd": "Gastos rentarios (USD)",
        "renta_variable_usd": "Renta Adicional (USD)",
        "rp_en_fact_usd": "Renta Porcentual (en Fact.) (USD)",
        "renta_minima_usd": "Renta Mínima (USD)",
        "cxc_saldo_usd": "CxC fin de mes (USD)",
        "fact_rent_core_usd": "Facturación Core (USD)",
        "promedio_pago_dias": "Prom. Pago (días)",
        "ar_ratio": "AR ratio (CxC/Core)",
        "garantia_monto_usd": "Garantía (USD)",
        "garantia_meses": "Garantía/CxC",
        "score_composite": "Score",
        "rating": "Rating",
        "sentinel_score": "Puntaje Sentinel",
        "pd_annual": "PD Anual",
        "pd_monthly": "PD Mensual",
    })

    nice_cols = [
        "Fecha", "Razón Social", "Nombre Comercial",
        "Ventas (USD)", "Gastos rentarios (USD)", "Costo de Ocupación (%)",
        "Renta Mínima (USD)", "Renta Porcentual (en Fact.) (USD)", "Renta Adicional (USD)",
        "CxC fin de mes (USD)", "Facturación Core (USD)",
        "Prom. Pago (días)", "AR ratio (CxC/Core)",
        "Garantía (USD)", "Garantía/CxC", "Puntaje Sentinel",
        "Score", "Rating", "PD Anual", "PD Mensual",
    ]
    nice_cols = [c for c in nice_cols if c in nice_es.columns]

    rr_keys = {normalize_group_key(x) for x in CONFIG["rent_related_groups"]}
    breakdown = (
        fact_grouped[fact_grouped["NomGrupoKey"].isin(rr_keys)]
        .rename(columns={"Razon Social": "Razón Social", "NomGrupo": "Grupo", "usd": "USD"})
        .sort_values(["Razón Social", "Fecha", "Grupo"])
    )

    xlsx_path = os.path.join(OUTPUT_DIR, "tenant_month_ratings_es.xlsx")
    with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as xl:
        nice_es[nice_cols].sort_values(["Razón Social", "Fecha"]).to_excel(
            xl, sheet_name="Resumen (ES)", index=False
        )
        base[raw_cols].sort_values(["Razon Social", "Fecha"]).to_excel(
            xl, sheet_name="Ratings (Raw)", index=False
        )
        breakdown.to_excel(xl, sheet_name="Rent-Related Breakdown", index=False)

    print("\nSaved:")
    print(" -", raw_path)
    print(" -", xlsx_path)

    # Basic analytics export
    analytics_xlsx = os.path.join(OUTPUT_DIR, "score_analytics.xlsx")
    try:
        df_ana = base.copy()

        metrics_for_bands = [
            "promedio_pago_dias", "ar_ratio", "occ_cost", "cxc_saldo_usd",
            "ventas_usd", "facturacion_usd", "sentinel_score", "pd_annual", "pd_monthly",
        ]
        available_metrics = [m for m in metrics_for_bands if m in df_ana.columns]

        band_summary = pd.DataFrame()
        if "rating" in df_ana.columns and available_metrics:
            band_summary = df_ana.groupby("rating")[available_metrics].mean().sort_index()

        corr_cols = [
            "score_composite", "promedio_pago_dias", "ar_ratio", "occ_cost", "sentinel_score"
        ]
        corr_cols = [c for c in corr_cols if c in df_ana.columns]
        corr_matrix = df_ana[corr_cols].corr() if len(corr_cols) >= 2 else pd.DataFrame()

        rating_year = pd.DataFrame()
        if "rating" in df_ana.columns and "Fecha" in df_ana.columns:
            df_ana["year"] = pd.to_datetime(df_ana["Fecha"], errors="coerce").dt.year
            rating_year = (
                df_ana.pivot_table(
                    index="rating",
                    columns="year",
                    values="Razon Social",
                    aggfunc="nunique",
                )
                .fillna(0)
                .astype(int)
                .sort_index()
            )

        with pd.ExcelWriter(analytics_xlsx, engine="xlsxwriter") as xl:
            df_ana.to_excel(xl, sheet_name="All Data", index=False)
            if not band_summary.empty:
                band_summary.to_excel(xl, sheet_name="Band Summary")
            if not corr_matrix.empty:
                corr_matrix.to_excel(xl, sheet_name="Correlations")
            if not rating_year.empty:
                rating_year.to_excel(xl, sheet_name="Rating by Year")

        print(" -", analytics_xlsx)
    except Exception as e:
        print("\nCould not write analytics report:", e)


if __name__ == "__main__":
    run_pipeline()

RuntimeError: Could not read C:\OneDrive\OneDrive - Jockey Plaza\Documentos\Score Riesgo Locatarios\Inputs Dataset Score vf\Feed final (csv)\Facturacion 2023-2025.csv. Last error: [Errno 2] No such file or directory: 'C:\\OneDrive\\OneDrive - Jockey Plaza\\Documentos\\Score Riesgo Locatarios\\Inputs Dataset Score vf\\Feed final (csv)\\Facturacion 2023-2025.csv'